# ĐẶC TẢ RESTFUL API — HỆ THỐNG CẢNH BÁO SỚM AQI (v2.0)
## Hợp tác cùng Sở Tài nguyên & Môi trường TP — Đồng bộ theo User Stories & Gherkin BDD

*Tài liệu này là bản chỉnh sửa của đặc tả RESTful API gốc, được rà soát và đồng bộ lại theo đúng 11 User Story (US01–US11) và kịch bản Gherkin BDD trong file `UserStories_GherkinBDD.xlsx`.*

---

## 0. Tóm Tắt Các Thay Đổi Chính

So với bản đặc tả trước, tài liệu này điều chỉnh 3 nhóm vấn đề:

1. **Người dân KHÔNG cần đăng nhập ở bất kỳ đâu** (US06, US07, US09, US11): loại bỏ toàn bộ khái niệm tài khoản người dân, hồ sơ sức khỏe cá nhân, liên kết phụ huynh–con, và cơ chế gửi SMS/Push cá nhân hóa. Cảnh báo và khuyến nghị hành động nay hiển thị dưới dạng **banner công khai trên trang**, giống nhau cho mọi người xem, không phân biệt tài khoản.
2. **Khu vực Admin (Chuyên viên phân tích) tách biệt hoàn toàn** (US01): có màn hình đăng nhập admin riêng, không hiển thị với người dân; toàn bộ endpoint phân tích nội bộ nằm dưới namespace `/api/v1/admin/**` và bắt buộc xác thực Bearer token.
3. **Bổ sung nhóm API Admin Dashboard** (yêu cầu bổ sung): thêm 4 endpoint tổng quan (`/admin/dashboard/*`) để Chuyên viên phân tích xem nhanh dữ liệu tổng hợp toàn hệ thống mà không cần tự tổng hợp từ dữ liệu thô.

Các bảng `USERS` (cá nhân hóa), `HEALTH_CONDITIONS`, `USER_HEALTH_CONDITIONS`, `PARENT_CHILD_LINKS`, `ALERTS_SENT` (gắn `user_id`) của bản cũ đã được loại bỏ vì không có user story nào yêu cầu — thay bằng `ADMIN_ACCOUNTS` (chỉ cho Chuyên viên phân tích) và `ALERT_EVENTS` (log cảnh báo công khai theo trạm, không gắn người dùng cụ thể). Nếu nhóm dự án vẫn cần tính năng khai báo bệnh nền / SMS cá nhân hóa cho giai đoạn sau, cần bổ sung user story tương ứng trước khi đưa lại vào đặc tả.

### Bảng đối chiếu User Story ↔ API Endpoint

| User Story | Actor | Endpoint đáp ứng |
| :--- | :--- | :--- |
| US01 | Chuyên viên phân tích | `POST /api/v1/admin/auth/login` |
| US02 | Chuyên viên phân tích | `GET /api/v1/admin/readings` |
| US03 | Chuyên viên phân tích | `GET /api/v1/admin/analytics/heatmap`, `GET /api/v1/admin/analytics/hourly-trend`, `GET /api/v1/admin/analytics/windrose` |
| US04 | Chuyên viên phân tích | `GET /api/v1/admin/analytics/seasonal-comparison`, `GET /api/v1/admin/analytics/prominent-pollutants` |
| US05 | Chuyên viên phân tích | `POST /api/v1/admin/data-cleaning/impute` |
| US06 | Người dân | `GET /api/v1/public/stations` |
| US07 | Người dân | `GET /api/v1/public/aqi/current` |
| US08 | Người dân | `GET /api/v1/public/aqi/forecast` |
| US09 | Người dân | `GET /api/v1/public/alerts/active` (+ nhúng sẵn trong `aqi/current`) |
| US10 | Người dân | `GET /api/v1/public/aqi/history` |
| US11 | Người dân | Trường `recommended_action` trong `aqi/current`, `aqi/forecast`, `alerts/active` |
| *(Bổ sung theo yêu cầu)* | Admin | `GET /api/v1/admin/dashboard/overview`, `/alerts-summary`, `/model-performance`, `/data-quality` |

---

## 1. MAPPING MÔ HÌNH ĐỐI CHIẾU THỐNG NHẤT

### 1.1. Sáu Phân Lớp AQI Tiêu Chuẩn & Banner Cảnh Báo Công Khai

Bảng `AQI_LEVELS` không đổi về 6 phân lớp, nhưng cột "kích hoạt cảnh báo" và "khuyến nghị" được viết lại theo đúng cơ chế **banner công khai** (US09, US11) thay vì gửi thông báo cá nhân hóa:

| AQI Range | Class Name (ML) | Nhãn Tiếng Việt | Banner công khai kích hoạt (US09) | Khuyến nghị hành động — hiển thị cho mọi người xem (US11) |
| :--- | :--- | :--- | :--- | :--- |
| **0–50** | `Good` | Tốt | Không banner | Không khí trong lành, an toàn cho tất cả mọi người. |
| **51–100** | `Moderate` | Trung bình | Không banner | Chất lượng chấp nhận được; nhóm cực kỳ nhạy cảm nên lưu ý. |
| **101–150** | `Unhealthy for Sensitive Groups` | Kém | **"Cảnh báo nhóm nhạy cảm"** (AQI > 100) | Nhóm nhạy cảm (trẻ em, người già, người có bệnh hô hấp) nên hạn chế vận động mạnh ngoài trời. |
| **151–200** | `Unhealthy` | Xấu | **+ "Cảnh báo nguy hại"** hiển thị thêm (AQI > 150), trong vòng 5 phút | Nên đeo khẩu trang chuyên dụng khi ra ngoài. |
| **201–300** | `Very Unhealthy` | Rất xấu | **+ "Cảnh báo nguy hại"** | Hạn chế ra ngoài, đặc biệt người già và trẻ em. |
| **> 300** | `Hazardous` | Nguy hại | **+ "Cảnh báo nguy hại" mức cao** | Nên ở trong nhà và đóng kín cửa sổ. |

> **Lưu ý quan trọng (US09 Gherkin):** khi AQI > 150, hệ thống hiển thị **THÊM** banner "Cảnh báo nguy hại" bên cạnh banner "Cảnh báo nhóm nhạy cảm" đã có — tức 2 banner có thể cùng xuất hiện, banner này không thay thế banner kia. Xem response mẫu tại API `GET /api/v1/public/alerts/active` (mục 3, API 5).

---

## 2. KIẾN TRÚC DATABASE (REVISED)

### 2.1. Nhóm bảng Quan trắc & Dự báo (không đổi so với bản gốc)

*   **`STATIONS`**: `station_id` (PK), `name`, `latitude`, `longitude`, `district`, `status`.
*   **`AIR_QUALITY_READINGS`**: `reading_id` (PK), `station_id` (FK), `datetime`, `pm25`, `pm10`, `so2`, `no2`, `co`, `o3`, `temp`, `pres`, `dewp`, `rain`, `wd`, `wspm`.
*   **`DAILY_STATION_SUMMARY`**: `summary_id` (PK), `station_id` (FK), `date`, `avg_pm25`…`avg_o3`, `aqi_avg`, `season`.
*   **`AQI_FORECASTS`**: `forecast_id` (PK), `station_id` (FK), `model_id` (FK), `created_at`, `target_time`, `predicted_aqi`, `predicted_class`.
*   **`ML_MODELS`** *(định nghĩa rõ vì được `AQI_FORECASTS` tham chiếu qua `model_id`, bản gốc chưa nêu chi tiết)*: `model_id` (PK), `version`, `trained_at`, `accuracy`, `mae`, `rmse`, `status` ("Ready"/"Archived").

### 2.2. Nhóm bảng Quản trị & Cảnh báo (THAY ĐỔI so với bản gốc)

*   **`ADMIN_ACCOUNTS`** *(thay thế bảng `USERS` cũ — chỉ dành cho Chuyên viên phân tích, KHÔNG áp dụng cho người dân)* (US01):
    *   `admin_id` (INT, PK), `username` (VARCHAR, unique), `password_hash` (VARCHAR), `full_name` (VARCHAR), `role` (VARCHAR — "Chuyên viên phân tích"/"Quản trị viên"), `created_at`, `last_login_at`.
*   **`ALERT_EVENTS`** *(thay thế bảng `ALERTS_SENT` cũ — log banner công khai theo trạm, KHÔNG gắn `user_id`)* (US09):
    *   `alert_id` (BIGINT, PK), `station_id` (FK -> STATIONS), `forecast_id` (FK -> AQI_FORECASTS, nullable), `aqi_value` (FLOAT), `alert_level` (VARCHAR — "SensitiveGroup"/"Hazard"), `banner_message` (VARCHAR), `triggered_at` (DATETIME), `resolved_at` (DATETIME, nullable — thời điểm AQI giảm lại dưới ngưỡng, banner tự tắt).

*   ~~`USERS` (cá nhân hóa), `HEALTH_CONDITIONS`, `USER_HEALTH_CONDITIONS`, `PARENT_CHILD_LINKS`~~ — **đã loại bỏ**: không có user story nào yêu cầu khai báo bệnh nền hay tài khoản người dân (US07, US11 nêu rõ "không cần khai báo hồ sơ sức khỏe" / "không phân biệt tài khoản").

---

## 3. PHÂN HỆ CÔNG KHAI (`/api/v1/public/**`) — KHÔNG YÊU CẦU ĐĂNG NHẬP

> Toàn bộ endpoint trong mục này **không kèm header `Authorization`**, không yêu cầu tài khoản, không lưu bất kỳ thông tin cá nhân nào của người truy cập — đúng theo US06, US07, US09, US11.

#### API 1: Tìm kiếm / danh sách trạm quan trắc theo thành phố (US06)
* **Endpoint:** `GET /api/v1/public/stations`
* **Query Parameters:**
    * `q` (string, optional): từ khóa tìm kiếm tên trạm/thành phố, dùng cho gợi ý auto-complete (giống ứng dụng dự báo thời tiết). Bỏ trống để lấy toàn bộ danh sách trạm đang hoạt động.
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "data": [
        { "station_id": 1, "name": "Guanyuan", "district": "Ba Đình", "latitude": 21.0285, "longitude": 105.8542 },
        { "station_id": 2, "name": "Dongsi", "district": "Hoàn Kiếm", "latitude": 21.0333, "longitude": 105.8500 }
      ]
    }
    ```

#### API 2: AQI hiện tại của trạm/thành phố đã chọn (US07)
* **Endpoint:** `GET /api/v1/public/aqi/current`
* **Query Parameters:**
    * `station_id` (int, required)
* **Response (200 OK):** Luôn kèm ghi chú ngưỡng nhạy cảm và khuyến nghị mặc định cho mọi người xem — không cần khai báo hồ sơ sức khỏe.
    ```json
    {
      "success": true,
      "station": { "station_id": 1, "name": "Guanyuan" },
      "current_aqi": {
        "value": 118.5,
        "class_name": "Unhealthy for Sensitive Groups",
        "label": "Kém",
        "main_pollutant": "pm25",
        "measured_at": "2026-09-09T09:00:00Z"
      },
      "sensitive_group_note": "Ngưỡng an toàn cho nhóm nhạy cảm (trẻ em, người già, người có bệnh hô hấp): AQI > 100.",
      "recommended_action": "Nhóm nhạy cảm nên hạn chế vận động mạnh ngoài trời.",
      "active_alert_levels": ["SensitiveGroup"]
    }
    ```

#### API 3: Dự báo AQI 24–48 giờ tới (US08)
* **Endpoint:** `GET /api/v1/public/aqi/forecast`
* **Query Parameters:**
    * `station_id` (int, required)
    * `hours` (int, optional, mặc định 48, tối đa 48)
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "station_id": 1,
      "forecast": [
        { "target_time": "2026-09-09T10:00:00Z", "predicted_aqi": 122.0, "predicted_class": "Unhealthy for Sensitive Groups", "confidence_range": [110.5, 133.5] },
        { "target_time": "2026-09-09T18:00:00Z", "predicted_aqi": 162.0, "predicted_class": "Unhealthy", "confidence_range": [148.0, 176.0] }
      ]
    }
    ```
    > `confidence_range` là optional — trả về `null` nếu mô hình hiện tại chưa hỗ trợ khoảng tin cậy.

#### API 4: Lịch sử AQI tuần/tháng qua (US10)
* **Endpoint:** `GET /api/v1/public/aqi/history`
* **Query Parameters:**
    * `station_id` (int, required)
    * `range` (string, optional, mặc định `"7d"`): `"7d"` | `"30d"`, hoặc dùng `start_date`/`end_date` tùy chỉnh.
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "station_id": 1,
      "range": "7d",
      "history": [
        { "date": "2026-09-03", "aqi_avg": 95.0, "class_name": "Moderate" },
        { "date": "2026-09-04", "aqi_avg": 118.0, "class_name": "Unhealthy for Sensitive Groups" }
      ]
    }
    ```

#### API 5: Cảnh báo (banner) đang hoạt động của một trạm (US09, US11)
* **Endpoint:** `GET /api/v1/public/alerts/active`
* **Mục đích:** endpoint nhẹ để frontend poll định kỳ (khuyến nghị mỗi 1–5 phút) và cập nhật banner mà không phải tải lại toàn bộ trang, đáp ứng yêu cầu "trong vòng 5 phút kể từ khi có dự báo mới" của US09.
* **Query Parameters:**
    * `station_id` (int, required)
* **Response (200 OK) — ví dụ khi AQI dự báo = 162.0 (vượt cả 2 ngưỡng):**
    ```json
    {
      "success": true,
      "station_id": 1,
      "aqi_value": 162.0,
      "active_banners": [
        {
          "level": "SensitiveGroup",
          "message": "Cảnh báo nhóm nhạy cảm: AQI vượt ngưỡng 100.",
          "recommended_action": "Nhóm nhạy cảm nên hạn chế vận động mạnh ngoài trời."
        },
        {
          "level": "Hazard",
          "message": "Cảnh báo nguy hại: AQI vượt ngưỡng 150.",
          "recommended_action": "Nên đeo khẩu trang chuyên dụng khi ra ngoài."
        }
      ],
      "triggered_at": "2026-09-09T09:03:00Z"
    }
    ```
* **Response khi AQI ≤ 100:** `"active_banners": []` (không banner nào được hiển thị).

---

## 4. PHÂN HỆ QUẢN TRỊ (`/api/v1/admin/**`) — YÊU CẦU ĐĂNG NHẬP ADMIN

> Toàn bộ endpoint trong mục này bắt buộc header `Authorization: Bearer <admin_token>`, cấp phát sau khi đăng nhập thành công qua API 6. Namespace `/admin/**` **không xuất hiện** ở bất kỳ đâu trên trang công khai dành cho người dân (US01).

#### API 6: Đăng nhập khu vực Admin (US01)
* **Endpoint:** `POST /api/v1/admin/auth/login`
* **Request Body:**
    ```json
    { "username": "chuyenvien.a", "password": "********" }
    ```
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "access_token": "eyJhbGciOiJIUzI1NiIs...",
      "token_type": "Bearer",
      "expires_in": 3600,
      "admin": { "admin_id": 3, "full_name": "Nguyễn Văn A", "role": "Chuyên viên phân tích" }
    }
    ```
* **Response (401 Unauthorized):**
    ```json
    { "success": false, "message": "Sai tên đăng nhập hoặc mật khẩu." }
    ```

#### API 7: Xem, lọc và so sánh dữ liệu quan trắc theo trạm & thời gian (US02)
* **Endpoint:** `GET /api/v1/admin/readings`
* **Query Parameters:** `station_ids` (string, required — CSV, vd `"1,2"`), `start_date` (date, required), `end_date` (date, required)
* **Response (200 OK):** dữ liệu nhóm theo trạm để frontend vẽ biểu đồ song song cùng khung thời gian, cùng chỉ số.
    ```json
    {
      "success": true,
      "start_date": "2015-12-01",
      "end_date": "2016-02-28",
      "stations": [
        {
          "station_id": 1,
          "station_name": "Guanyuan",
          "readings": [
            { "datetime": "2015-12-01T00:00:00", "pm25": 88.0, "pm10": 120.0, "so2": 15.0, "no2": 45.0, "co": 1.2, "o3": 20.0, "temp": 2.1, "pres": 1020.5, "dewp": -5.3, "rain": 0.0, "wd": "NNE", "wspm": 3.2 }
          ]
        },
        { "station_id": 2, "station_name": "Dongsi", "readings": ["..."] }
      ]
    }
    ```
    > Với khoảng thời gian dài (nhiều tháng), khuyến nghị bổ sung phân trang (`page`, `page_size`) để tránh payload quá lớn.

#### API 8: Heatmap ô nhiễm theo tháng (US03)
* **Endpoint:** `GET /api/v1/admin/analytics/heatmap`
* **Query Parameters:** `pollutant` (required), `year` (required)
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "pollutant": "pm25",
      "year": 2015,
      "heatmap_matrix": [
        { "month": 12, "station_id": 1, "station_name": "Guanyuan", "avg_value": 84.5 },
        { "month": 12, "station_id": 2, "station_name": "Dongsi", "avg_value": 92.1 }
      ]
    }
    ```

#### API 9: Biểu đồ xu hướng theo giờ — Line chart (US03)
* **Endpoint:** `GET /api/v1/admin/analytics/hourly-trend`
* **Query Parameters:** `station_id` (required), `pollutant` (required), `date` (required, vd `"2016-01-15"`)
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "station_id": 1,
      "pollutant": "pm25",
      "date": "2016-01-15",
      "hourly_data": [
        { "hour": 0, "value": 45.2 },
        { "hour": 1, "value": 42.8 }
      ]
    }
    ```

#### API 10: Biểu đồ hoa hồng gió — Wind Rose (US03)
* **Endpoint:** `GET /api/v1/admin/analytics/windrose`
* **Query Parameters:** `station_id` (required), `pollutant` (required — dùng để phủ màu nồng độ), `start_date`, `end_date` (optional)
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "station_id": 1,
      "pollutant": "pm25",
      "wind_rose_data": [
        { "direction": "N", "frequency_pct": 6.2, "avg_speed": 2.1, "avg_pollutant_value": 55.3 },
        { "direction": "NNE", "frequency_pct": 8.5, "avg_speed": 3.4, "avg_pollutant_value": 88.0 }
      ],
      "dominant_high_pollution_direction": "NNE"
    }
    ```

#### API 11: So sánh nồng độ ô nhiễm trung bình giữa 4 mùa (US04)
* **Endpoint:** `GET /api/v1/admin/analytics/seasonal-comparison`
* **Query Parameters:** `station_id` (required), `pollutant` (required)
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "station_id": 1,
      "pollutant": "pm25",
      "seasonal_data": { "Spring": 48.5, "Summer": 32.4, "Autumn": 52.1, "Winter": 115.8 },
      "highest_pollution_season": "Winter"
    }
    ```

#### API 12: Xếp hạng chất ô nhiễm nổi bật theo mùa (US04)
* **Endpoint:** `GET /api/v1/admin/analytics/prominent-pollutants`
* **Query Parameters:** `season` (required)
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "season": "Winter",
      "ranking": [
        { "rank": 1, "pollutant": "pm25", "excess_rate": 1.58 },
        { "rank": 2, "pollutant": "pm10", "excess_rate": 1.12 }
      ],
      "caution_note": "Không kết luận chắc chắn về nguồn phát thải nếu thiếu dữ liệu giao thông/công nghiệp."
    }
    ```

#### API 13: Kích hoạt làm sạch & nội suy dữ liệu (US05)
* **Endpoint:** `POST /api/v1/admin/data-cleaning/impute`
* **Request Body:**
    ```json
    { "station_id": 1, "apply_interpolation": true }
    ```
* **Response (200 OK):** khoảng trống ≤ 3 giờ liên tiếp được nội suy tuyến tính tự động; > 3 giờ giữ nguyên, đánh dấu để xem xét thủ công.
    ```json
    {
      "success": true,
      "station_id": 1,
      "summary": {
        "interpolated_records_count": 42,
        "marked_for_manual_review_count": 12,
        "details": "Nội suy tuyến tính thành công cho các khoảng trống <= 3h liên tiếp."
      }
    }
    ```

---

## 5. ADMIN DASHBOARD — DỮ LIỆU TỔNG QUAN (`/api/v1/admin/dashboard/**`)

> Nhóm API mới, bổ sung theo yêu cầu: cho phép Chuyên viên phân tích / Quản trị viên xem nhanh dữ liệu tổng hợp toàn hệ thống mà không cần tự truy vấn dữ liệu thô. Cùng yêu cầu Bearer token như mục 4.

#### API 14: Tổng quan hệ thống
* **Endpoint:** `GET /api/v1/admin/dashboard/overview`
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "generated_at": "2026-09-09T10:00:00Z",
      "stations_summary": { "total": 12, "active": 11, "inactive": 1 },
      "citywide_avg_aqi": 96.4,
      "highest_aqi_station": { "station_id": 1, "name": "Guanyuan", "aqi_value": 162.0 },
      "lowest_aqi_station": { "station_id": 5, "name": "Aotizhongxin", "aqi_value": 42.0 },
      "stations_in_alert": 3
    }
    ```

#### API 15: Thống kê lịch sử cảnh báo
* **Endpoint:** `GET /api/v1/admin/dashboard/alerts-summary`
* **Query Parameters:** `start_date`, `end_date` (optional, mặc định 30 ngày gần nhất)
* **Response (200 OK):** tổng hợp trực tiếp từ bảng `ALERT_EVENTS`.
    ```json
    {
      "success": true,
      "period": { "start_date": "2026-08-10", "end_date": "2026-09-09" },
      "total_alerts": 47,
      "by_level": { "SensitiveGroup": 30, "Hazard": 17 },
      "by_station": [
        { "station_id": 1, "name": "Guanyuan", "count": 12 }
      ],
      "daily_timeline": [
        { "date": "2026-08-10", "count": 2 }
      ]
    }
    ```

#### API 16: Hiệu suất mô hình Machine Learning
* **Endpoint:** `GET /api/v1/admin/dashboard/model-performance`
* **Response (200 OK):**
    ```json
    {
      "success": true,
      "active_model": { "model_id": 15, "version": "v3.2.1", "trained_at": "2026-09-05T00:00:00Z", "accuracy": 0.7670, "mae": 12.45, "rmse": 18.12 },
      "model_history": [
        { "model_id": 14, "version": "v3.2.0", "accuracy": 0.7512, "trained_at": "2026-08-01T00:00:00Z" }
      ]
    }
    ```

#### API 17: Chất lượng dữ liệu tổng quan
* **Endpoint:** `GET /api/v1/admin/dashboard/data-quality`
* **Response (200 OK):** tổng hợp kết quả nội suy (API 13) trên toàn bộ 12 trạm.
    ```json
    {
      "success": true,
      "as_of": "2026-09-09",
      "stations": [
        { "station_id": 1, "name": "Guanyuan", "total_records": 17520, "interpolated_count": 42, "manual_review_count": 12, "completeness_pct": 99.7 }
      ]
    }
    ```

---

## 6. GIAO THỨC TÍCH HỢP MACHINE LEARNING (ML → BACKEND)

Giữ nguyên cơ chế push nội bộ giữa mô hình ML và Backend so với bản gốc — đây là giao tiếp service-to-service, không liên quan đến người dân hay xác thực admin.

### 6.1. Đăng ký metadata & kiểm định chất lượng mô hình
* **Endpoint:** `POST /api/v1/internal/ml-models/register`
* **Headers:** `X-ML-Service-Token: <secure_token>`
* **Request Body:** *(ví dụ rút gọn — cấu trúc đầy đủ gồm cả 6 phân lớp trong `classification_report`)*
    ```json
    {
      "version": "v3.2.1",
      "trained_at": "2026-09-05T00:00:00Z",
      "model_metrics": {
        "rows_used": 412028,
        "training_rows": 329622,
        "testing_rows": 82406,
        "accuracy": 0.7670072567531491,
        "mae": 12.45,
        "rmse": 18.12
      },
      "classification_report": {
        "Good": { "precision": 0.8349, "recall": 0.7677, "f1-score": 0.7999, "support": 13084.0 },
        "Unhealthy for Sensitive Groups": { "precision": 0.6275, "recall": 0.4857, "f1-score": 0.5476, "support": 10961.0 }
      }
    }
    ```
* **Response (201 Created):**
    ```json
    { "success": true, "model_id": 15, "status": "Ready", "message": "Đăng ký phiên bản mô hình thành công. Sẵn sàng nhận dự báo." }
    ```

### 6.2. Đẩy dự báo hàng giờ từ ML sang Backend
* **Endpoint:** `POST /api/v1/internal/ml-predictions`
* **Headers:** `X-ML-Service-Token: <secure_token>`
* **Request Body:**
    ```json
    {
      "model_id": 15,
      "generated_at": "2026-09-09T09:00:00Z",
      "forecasts": [
        { "station_id": 1, "target_time": "2026-09-09T18:00:00Z", "predicted_aqi": 162.0, "predicted_class": "Unhealthy" }
      ]
    }
    ```
* **Response (200 OK):** `alerts_triggered_count` nay phản ánh số bản ghi mới trong `ALERT_EVENTS` (banner công khai) — **không còn là số thông báo cá nhân đã gửi**.
    ```json
    {
      "success": true,
      "received_records": 1,
      "alerts_triggered_count": 2,
      "message": "Dữ liệu dự báo được ghi nhận. Đã cập nhật banner cảnh báo công khai cho trạm liên quan."
    }
    ```

---

## 7. QUY TRÌNH XỬ LÝ SỰ KIỆN CẢNH BÁO (ALERT PIPELINE — REVISED)

```
        [ ML Service ] định kỳ POST /internal/ml-predictions về Backend
                                │
                                ▼
             Hệ thống lưu dự báo vào bảng AQI_FORECASTS
                                │
                                ▼
              So khớp predicted_aqi với ngưỡng trong AQI_LEVELS
                                │
              ┌─────────────────┴─────────────────┐
              ▼                                    ▼
     AQI > 100 (Kém trở lên)               AQI > 150 (Xấu trở lên)
     → Ghi ALERT_EVENTS                    → Ghi THÊM ALERT_EVENTS
       level = "SensitiveGroup"              level = "Hazard"
              └─────────────────┬─────────────────┘
                                ▼
        Cập nhật banner công khai cho trạm — trong vòng 5 phút (US09)
                                │
                                ▼
   Người dân mở trang / gọi GET /public/alerts/active (KHÔNG cần đăng nhập)
        → thấy toàn bộ banner đang hoạt động + khuyến nghị hành động
                                │
                                ▼
        Admin xem lại lịch sử qua GET /admin/dashboard/alerts-summary
```

---

## 8. KIỂM THỬ NGHIỆM THU BDD (GHERKIN SYNTAX — REVISED)

### Kịch bản 1: Người dân xem AQI công khai, không cần tài khoản (US06, US07, US11)
```gherkin
Feature: Trang chất lượng không khí công khai cho người dân

  Scenario: Tìm trạm và xem AQI hiện tại kèm khuyến nghị mà không cần đăng nhập
    Given Danh sách 12 trạm quan trắc đã có sẵn trong hệ thống
    And Người dùng chưa đăng nhập và chưa khai báo bất kỳ thông tin cá nhân nào
    When Người dùng gọi GET /api/v1/public/stations?q="Guanyuan" mà không kèm header Authorization
    And Người dùng chọn trạm "Guanyuan" và gọi GET /api/v1/public/aqi/current?station_id=1
    Then Hệ thống trả về mã 200 OK cùng chỉ số AQI hiện tại, mức phân loại, ngưỡng nhạy cảm (AQI > 100) và ít nhất 1 khuyến nghị hành động
    And Toàn bộ luồng trên không yêu cầu đăng nhập hay khai báo hồ sơ sức khỏe
```

### Kịch bản 2: Banner cảnh báo công khai khi AQI dự báo vượt ngưỡng (US09)
```gherkin
Feature: Cảnh báo tự động dạng banner khi AQI dự báo vượt ngưỡng nguy hại

  Scenario: Hiển thị đồng thời 2 banner khi AQI dự báo vượt ngưỡng 150
    Given Mô hình dự báo (Model ID: 15) sinh AQI dự báo = 162.0 (Class: "Unhealthy") cho trạm số 1 (Guanyuan)
    When Backend nhận dữ liệu qua POST /api/v1/internal/ml-predictions và ghi vào bảng AQI_FORECASTS
    Then Hệ thống tạo 2 bản ghi trong bảng ALERT_EVENTS cho trạm số 1: level "SensitiveGroup" và level "Hazard"
    And Trong vòng 5 phút, bất kỳ ai gọi GET /api/v1/public/alerts/active?station_id=1 mà không cần đăng nhập đều nhận được cả 2 banner kèm khuyến nghị hành động tương ứng
    And Nội dung cảnh báo giống nhau cho mọi người xem, không cá nhân hóa theo tài khoản
```

### Kịch bản 3: Khu vực Admin tách biệt hoàn toàn khỏi trang công khai (US01)
```gherkin
Feature: Đăng nhập khu vực quản trị dành riêng cho Chuyên viên phân tích

  Scenario: Chỉ Chuyên viên phân tích đã đăng nhập mới truy cập được công cụ phân tích nội bộ
    Given Chuyên viên phân tích đã được Sở TN&MT cấp tài khoản (username/password)
    And Trang công khai dành cho người dân không hiển thị mục đăng nhập này
    When Chuyên viên phân tích gọi POST /api/v1/admin/auth/login với thông tin hợp lệ
    Then Hệ thống trả về access_token và cho phép truy cập các endpoint dưới /api/v1/admin/**
    But Khi gọi GET /api/v1/admin/analytics/heatmap mà không kèm access_token hợp lệ, hệ thống trả về 401 Unauthorized và từ chối truy cập
```